# La parte del blocco dove stanno quasi tutti i parametri

*Anatomia di un LLM* · capitolo 5, L'attenzione e il blocco · script `03-07`, `scripts/03_07_ffn_attivazioni.py`

Dopo l'attenzione ogni parola passa da sola in una piccola rete: sale in una
dimensione piu' grande, attraversa una non linearita', torna giu'. E' meta' del
modello in numero di parametri, ed e' anche il posto dove la scelta della
funzione fa la differenza.

Si esegue con `python3 scripts/03_07_ffn_attivazioni.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/03_07_ffn_attivazioni.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [1]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [2]:
import torch
import torch.nn.functional as F

from llm_anatomy.config import AgoGPTConfig
from llm_anatomy.model import FFN, SwiGLU
from llm_anatomy.utils import (conta_parametri, dec, numero, prepara, stampa_shape,
                               tabella, titolo, verifica)

prepara()

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


device(type='cpu')

In [3]:
titolo("Tre funzioni a confronto")
x = torch.tensor([-3.0, -1.0, -0.5, 0.0, 0.5, 1.0, 3.0])
righe = []
for v in x.tolist():
    t = torch.tensor(v)
    righe.append([dec(v, 1), dec(F.relu(t), 4), dec(F.gelu(t), 4),
                  dec(F.silu(t), 4)])
tabella(["x", "ReLU", "GELU", "SiLU"], righe, allinea="rrrr")
print("  ReLU azzera di netto, GELU e SiLU lasciano passare un poco di")
print("  negativo: la derivata non e' mai esattamente zero, e un neurone")
print("  spento puo' tornare a vivere")


Tre funzioni a confronto
------------------------
   x    ReLU     GELU     SiLU
----  ------  -------  -------
-3,0  0,0000  -0,0040  -0,1423
-1,0  0,0000  -0,1587  -0,2689
-0,5  0,0000  -0,1543  -0,1888
 0,0  0,0000   0,0000   0,0000
 0,5  0,5000   0,3457   0,3112
 1,0  1,0000   0,8413   0,7311
 3,0  3,0000   2,9960   2,8577
  ReLU azzera di netto, GELU e SiLU lasciano passare un poco di
  negativo: la derivata non e' mai esattamente zero, e un neurone
  spento puo' tornare a vivere


In [4]:
titolo("Quello che le distingue davvero")
for nome, fn in (("ReLU", F.relu), ("GELU", F.gelu), ("SiLU", F.silu)):
    t = torch.tensor(0.0, requires_grad=True)
    fn(t).backward()
    t2 = torch.tensor(-2.0, requires_grad=True)
    fn(t2).backward()
    print(f"  {nome:<6} f'(0) = {t.grad.item():7.4f}   "
          f"f'(-2) = {t2.grad.item():8.5f}")


Quello che le distingue davvero
-------------------------------
  ReLU   f'(0) =  0.0000   f'(-2) =  0.00000
  GELU   f'(0) =  0.5000   f'(-2) = -0.08523
  SiLU   f'(0) =  0.5000   f'(-2) = -0.09078


In [5]:
titolo("La versione classica: due matrici")
d = 512
cfg = AgoGPTConfig(d_model=d, n_head=8, act="gelu", vocab_size=93)
ffn = FFN(cfg)
print(f"  d_model {d}, d_ff {cfg.d_ff} = 4 x {d}")
x = torch.randn(2, 8, d)
stampa_shape("ingresso", x)
stampa_shape("dopo la salita", ffn.w_up(x))
stampa_shape("uscita", ffn(x))
print(f"  parametri: {conta_parametri(ffn)} = 2 x {d} x {cfg.d_ff}")


La versione classica: due matrici
---------------------------------
  d_model 512, d_ff 2048 = 4 x 512
  ingresso                     (2, 8, 512)            float32
  dopo la salita               (2, 8, 2048)           float32
  uscita                       (2, 8, 512)            float32
  parametri: 2097152 = 2 x 512 x 2048


In [6]:
titolo("SwiGLU: tre matrici, e una porta")
cfg_sw = AgoGPTConfig(d_model=d, n_head=8, act="swiglu", vocab_size=93)
sw = SwiGLU(cfg_sw)
print(f"  d_ff scelta dalla configurazione: {cfg_sw.d_ff}")
print(f"  8/3 x {d} = {dec(8 * d / 3, 1)}, arrotondato al multiplo di 32")
stampa_shape("porta", sw.w_gate(x))
stampa_shape("salita", sw.w_up(x))
stampa_shape("uscita", sw(x))


SwiGLU: tre matrici, e una porta
--------------------------------
  d_ff scelta dalla configurazione: 1376
  8/3 x 512 = 1365,3, arrotondato al multiplo di 32
  porta                        (2, 8, 1376)           float32
  salita                       (2, 8, 1376)           float32
  uscita                       (2, 8, 512)            float32


In [7]:
titolo("Perche' otto terzi")
righe = []
for nome, n_matrici, fattore in (("FFN classica", 2, 4.0),
                                 ("SwiGLU a 4d", 3, 4.0),
                                 ("SwiGLU a 8/3 d", 3, 8 / 3)):
    d_ff = int(fattore * d)
    par = n_matrici * d * d_ff
    classica = 2 * d * 4 * d
    righe.append([nome, n_matrici, d_ff, numero(par), dec(par / classica)])
tabella(["variante", "matrici", "d_ff", "parametri", "rispetto alla classica"],
        righe)
verifica(abs(3 * (8 / 3) - 2 * 4) < 1e-9,
         "tre matrici di larghezza 8/3 d contano quanto due di larghezza 4d")
print("  la sostituzione e' quindi gratuita in parametri e in calcolo: si")
print("  paga solo una moltiplicazione elemento per elemento in piu'")


Perche' otto terzi
------------------
variante        matrici  d_ff  parametri  rispetto alla classica
--------------  -------  ----  ---------  ----------------------
FFN classica          2  2048  2.097.152                    1,00
SwiGLU a 4d           3  2048  3.145.728                    1,50
SwiGLU a 8/3 d        3  1365  2.096.640                    1,00
  [verifica superata] tre matrici di larghezza 8/3 d contano quanto due di
                      larghezza 4d
  la sostituzione e' quindi gratuita in parametri e in calcolo: si
  paga solo una moltiplicazione elemento per elemento in piu'


In [8]:
titolo("Sul nostro modello")
righe = []
for act in ("relu", "gelu", "swiglu"):
    c = AgoGPTConfig(d_model=192, n_head=6, act=act, vocab_size=93)
    modulo = SwiGLU(c) if act == "swiglu" else FFN(c)
    righe.append([act, c.d_ff, conta_parametri(modulo)])
tabella(["attivazione", "d_ff", "parametri della FFN"], righe)


Sul nostro modello
------------------
attivazione  d_ff  parametri della FFN
-----------  ----  -------------------
relu          768               294912
gelu          768               294912
swiglu        512               294912


## Da provare

lascia d_ff a 4*d con SwiGLU e guarda i parametri crescere del 50%.